# 08 · Skills · MCP · Custom Agents · Slash Commands

这四件套**直接对位**你 `cowork_worker` 里最复杂的几块：

| 现状 | SDK 替代 |
|---|---|
| `SkillsContextProvider` + `<available_skills>` 系统块 + Cosmos `enabled_skills` | **Skills** —— `skill_directories=` + `disabled_skills=` |
| 外部接 Bing / 数据库 / GitHub MCP | **MCP servers** —— `mcp_servers={...}` |
| 主 agent 内联所有 prompt + 所有工具 | **Custom agents** —— 主 agent 仅编排，把脏活分给 sub-agent |
| `/help`、`/reset` 等控制指令现在走自己的 backend 路由 | **Commands** —— `commands=[CommandDefinition(...)]` |

## 0. 扩展性四件套全景

这 4 个 `create_session(...)` 入参是 SDK 给你的「主入口」：

```mermaid
flowchart TB
    Session["create_session(...)"]

    subgraph S1["skill_directories=[...]"]
        SK1["skills/<br/>├── translate-zh/SKILL.md<br/>└── code-review/SKILL.md"]
        SK1 --> SkillNote["CLI 扫子目录<br/>把 SKILL.md 内容注入 context<br/>(disabled_skills= 可关闭某些)"]
    end

    subgraph S2["mcp_servers={...}"]
        M1["stdio MCP<br/>(npx @modelcontextprotocol/server-fs)"]
        M2["http/sse MCP<br/>(learn.microsoft.com/api/mcp)"]
        M1 --> MCPNote["CLI 自动 spawn + tools/list + tools/call<br/>tools=['*'] 暴露全部，或白名单"]
        M2 --> MCPNote
    end

    subgraph S3["custom_agents=[...]"]
        CA1["researcher<br/>(只读: view+grep+find_files)"]
        CA2["editor<br/>(写: view+edit_file+bash)"]
        CA1 & CA2 --> CANote["主 agent 按 description 自动委派<br/>(subagent.* 事件)<br/>每个 sub-agent 自带工具白名单 + 可选 skills + MCP"]
    end

    subgraph S4["commands=[...]"]
        CMD1["/status, /dump ..."]
        CMD1 --> CMDNote["用户输 /xxx<br/>路由到你的 handler(ctx)"]
    end

    Session --> S1
    Session --> S2
    Session --> S3
    Session --> S4
```

### 对位 cowork_worker 现状

| cowork_worker 现在 | SDK 替代 | 改 / 删 |
|---|---|---|
| `SkillsContextProvider.before_run` 拼 `<available_skills>` | `skill_directories=` + `disabled_skills=` | **删** SkillsContextProvider |
| 接 Bing / GitHub / 内部 API → 写自定义 tool | `mcp_servers={'bing':..., 'gh':..., 'internal':...}` | 重写为 MCP（社区已有大量现成 MCP server） |
| 一个大 agent 装所有工具 + 所有 system prompt | 拆成 `researcher` / `editor` / `runner` 等 `custom_agents` | **改**：主 agent 当编排，sub-agent 干活 |
| 前端 `/reset` `/help` 自己拦 + 路由 | `commands=[CommandDefinition(...)]` | **改**（可选）|

下面 §1-§4 逐一演示，§5 给出**全部 4 个一起上**的样板代码——那就是迁完后的 cowork_worker 主入口长相。


In [ ]:
%pip install -q github-copilot-sdk python-dotenv

In [ ]:
import os, asyncio
from pathlib import Path
from dotenv import load_dotenv
from copilot import CopilotClient
from copilot.session import PermissionHandler, CommandDefinition, CommandContext
from copilot.generated.session_events import AssistantMessageData, SessionIdleData

load_dotenv()
AZURE = {
    'type': 'azure',
    'base_url': os.environ['AZURE_OPENAI_ENDPOINT'],
    'api_key': os.environ['AZURE_OPENAI_API_KEY'],
    'azure': {'api_version': os.getenv('AZURE_OPENAI_API_VERSION', '2024-10-21')},
}
MODEL = os.environ['AZURE_OPENAI_DEPLOYMENT']

async def wait_idle(session):
    done = asyncio.Event(); out = []
    def on_event(e):
        match e.data:
            case AssistantMessageData() as d: out.append(d.content)
            case SessionIdleData(): done.set()
    unsub = session.on(on_event); await done.wait(); unsub()
    return '\n'.join(out)

## 1. Skills —— 文件系统就是协议

**Skill = 目录 + `SKILL.md`**（带可选 YAML frontmatter）。指 `skill_directories=` 后，CLI 扫描子目录，把 `SKILL.md` 内容注入 context。

```
skills/
├── code-review/SKILL.md
└── translate-zh/SKILL.md
```

比你现在 `SkillsContextProvider.before_run` 拼 `<available_skills>` 块的方案**更省事**：CLI 自己负责加载、enable/disable、按需注入。

In [ ]:
# 落地两个 skill 到临时目录
SKILLS_DIR = Path('/tmp/cowork-sdk-skills')
SKILLS_DIR.mkdir(parents=True, exist_ok=True)

(SKILLS_DIR / 'translate-zh').mkdir(exist_ok=True)
(SKILLS_DIR / 'translate-zh' / 'SKILL.md').write_text('''---
name: translate-zh
description: Translate English text into idiomatic Simplified Chinese
---

# Translate to Chinese

When asked to translate, output ONLY the Chinese translation, no commentary.
Preserve markdown formatting. Prefer 地道中文 over literal translation.
''')

(SKILLS_DIR / 'caveman').mkdir(exist_ok=True)
(SKILLS_DIR / 'caveman' / 'SKILL.md').write_text('''---
name: caveman
description: Answer in ultra-terse caveman style
---

# Caveman mode

Drop articles and pleasantries. Use 4-8 word fragments. ALL CAPS for emphasis.
''')

print(list(SKILLS_DIR.iterdir()))

In [ ]:
async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
        skill_directories=[str(SKILLS_DIR)],
        disabled_skills=[],  # 全部启用
    ) as s:
        await s.send('Use caveman skill: explain what a Kubernetes pod is.')
        print(await wait_idle(s))
        await s.send('Now use translate-zh skill on your previous answer.')
        print(await wait_idle(s))

**对照 cowork_worker**：

- 你的 `enabled_skills` 列在 Cosmos workspace 文档里 → 用 `disabled_skills=[s for s in all if s not in enabled]` 转换
- `skill_directories` 可以指向你 `skills/` 仓库子目录
- 不再需要 `SkillsContextProvider` 这个抽象——SDK 自带

## 2. MCP servers —— 一等公民集成

支持两种：
- **local/stdio**：起子进程，stdio 通讯（npx、python -m、本地二进制）
- **http/sse**：远程服务

下面用官方 filesystem MCP 演示（需 npx 可用）。`tools=["*"]` 暴露全部工具，`[]` 禁全部，列表则白名单。

In [ ]:
MCP_DIR = Path('/tmp/cowork-sdk-mcp')
MCP_DIR.mkdir(exist_ok=True)
(MCP_DIR / 'hello.txt').write_text('Hello from MCP filesystem.')

mcp_servers = {
    'fs': {
        'type': 'local',
        'command': 'npx',
        'args': ['-y', '@modelcontextprotocol/server-filesystem', str(MCP_DIR)],
        'tools': ['*'],
        'timeout': 30000,
    },
    # 示例：远程 GitHub MCP（需要 token）
    # 'github': {
    #     'type': 'http',
    #     'url': 'https://api.githubcopilot.com/mcp/',
    #     'headers': {'Authorization': f'Bearer {os.environ["GH_TOKEN"]}'},
    #     'tools': ['*'],
    # },
}

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
        mcp_servers=mcp_servers,
    ) as s:
        await s.send(f'Use the fs MCP server to list files in {MCP_DIR} and read hello.txt.')
        print(await wait_idle(s))

**对照 cowork_worker**：
- Bing 搜索 → 找一个 Bing MCP server（社区有多个）
- 你 backend 自己的 REST API → 写一个轻量 stdio MCP 包一层 → agent 自然能调
- Foundry 现在的 hosted code interpreter → 暂时用内建 `bash` + `python_tool` 替代

## 3. Custom Agents —— 主 agent 当编排，sub-agent 干活

这是 SDK 最强的一块。每个 custom agent 自带 prompt + 工具白名单 + 可选 MCP + 可选 skills。runtime 按 `description` **自动匹配** 用户意图，分派给 sub-agent，结果合并回主 session。

经典模式：**researcher + editor**。

### 3.0 主 agent ↔ sub-agent 调度时序

`custom_agents` 不是简单的"prompt 模板列表"。runtime 会按每个 agent 的 `description` **匹配用户意图**自动分派，sub-agent 在独立子 session 里跑（带自己的 tools / skills / MCP），最终 result 合并回主对话：

```mermaid
sequenceDiagram
    autonumber
    participant User
    participant Main as 主 agent<br/>(默认)
    participant CLI as copilot CLI
    participant Sub1 as researcher<br/>(子 session)
    participant Sub2 as editor<br/>(子 session)

    User->>Main: "找到 print 改成 hello world，跑 pytest"
    Main->>CLI: 决定调 task(name=researcher, ...)
    CLI-->>User: event: subagent.started (researcher)
    activate Sub1
    Sub1->>Sub1: view + grep 找到 print
    Sub1-->>CLI: 返回结果 (file:line)
    deactivate Sub1
    CLI-->>User: event: subagent.completed
    CLI->>Main: 把 researcher 结果回灌

    Main->>CLI: 决定调 task(name=editor, ...)
    CLI-->>User: event: subagent.started (editor)
    activate Sub2
    Sub2->>Sub2: edit_file + bash (pytest)
    Sub2-->>CLI: 返回结果
    deactivate Sub2
    CLI-->>User: event: subagent.completed
    CLI->>Main: 把 editor 结果回灌
    Main-->>User: 最终汇总回复
```

**关键设计原则**：

- 主 agent **不应**直接持有重 context 的工具（用 `default_agent={'excluded_tools': [...]}` 强制隐藏）→ 它就只能委派
- sub-agent **只**装自己角色需要的工具 → context 干净、token 省
- 多角色场景：现在 `PER_TURN_MODEL` 切换 → 干脆给每个角色起独立 session（runtime 暂不支持 per-agent model）


In [ ]:
custom_agents = [
    {
        'name': 'researcher',
        'displayName': 'Code Researcher',
        'description': 'Read-only exploration: find files, grep for patterns, summarize code structure. NEVER modifies anything.',
        'tools': ['view', 'grep', 'find_files'],
        'prompt': 'You analyze code. Output a concise summary with file:line references. Do NOT modify files.',
    },
    {
        'name': 'editor',
        'displayName': 'Code Editor',
        'description': 'Makes minimal, surgical code changes. Verifies by running tests.',
        'tools': ['view', 'edit_file', 'bash'],
        'prompt': 'You make precise edits. Always verify by running tests via bash. Keep diffs minimal.',
    },
]

async with CopilotClient() as client:
    async with await client.create_session(
        on_permission_request=PermissionHandler.approve_all,
        model=MODEL, provider=AZURE,
        custom_agents=custom_agents,
    ) as s:
        os.chdir(s.workspace_path)
        # 观察 subagent.* 事件
        def tap(e):
            if e.type.startswith('subagent.'):
                print(f'  ◀ {e.type} {getattr(e.data, "agentName", "")}')
        s.on(tap)
        await s.send('Create app.py printing "hi", then have the researcher locate the print, '
                     'then have the editor change it to print "hello world". Run python app.py at the end.')
        print('\n===\n', await wait_idle(s))

### 3.1 给 sub-agent 单独装 MCP + skills

In [ ]:
specialized_agents = [
    {
        'name': 'security-auditor',
        'description': 'OWASP-focused code review',
        'prompt': 'Focus on OWASP Top 10. Cite CWE numbers.',
        'skills': ['translate-zh'],  # 这些 skill 会在该 sub-agent 启动时预注入
        'tools': ['view', 'grep'],
    },
]
# 在 create_session 里同时传 skill_directories + custom_agents 即可

### 3.2 `defaultAgent.excludedTools` —— 强制主 agent 委派

把某个**重 context** 的工具从主 agent 隐藏，只留给 sub-agent。这样主 agent 必须分派，保持自己 context 干净。Python 字段名为 `default_agent={'excluded_tools': [...]}`。

**对照 cowork_worker**：

- 你现在一个大 agent 装所有工具 → 拆成 `researcher`(只读) / `editor`(写文件) / `runner`(执行) / `summarizer`(总结)
- Cosmos 里 `enabled_skills` 不再是 system block 拼接 → 直接喂给某个 sub-agent 的 `skills:` 字段
- 多模型场景：现在 `PER_TURN_MODEL` 切换 → 干脆给每个角色不同 model（runtime 不支持 per-agent model；目前只能在 session 级，但**可以**对不同任务起不同 session）

## 4. Slash Commands —— 用户输 `/xxx` 路由到你的 handler

可以替代你 frontend 现在自己拦的 `/reset` / `/help` 等指令。

In [ ]:
async def handle_status(ctx: CommandContext) -> None:
    print(f'[status] session={ctx.session_id} args={ctx.args!r}')

async def handle_dump(ctx: CommandContext) -> None:
    print(f'[dump] would dump workspace for {ctx.session_id}')

commands = [
    CommandDefinition(name='status', description='Show session status', handler=handle_status),
    CommandDefinition(name='dump',   description='Dump workspace contents', handler=handle_dump),
]

# 用法：
# async with await client.create_session(..., commands=commands) as s:
#     await s.send('/status verbose')  # → 触发 handle_status，ctx.args='verbose'

## 5. 综合：skills + MCP + custom_agents + commands 一起上

下面这个 session config 就是你迁完后的 cowork_worker 主入口长相 —— **比现在简短 5x**。

In [ ]:
# 伪代码，组合所有特性
session_kwargs = dict(
    on_permission_request=PermissionHandler.approve_all,
    model=MODEL,
    provider=AZURE,
    skill_directories=[str(SKILLS_DIR)],
    disabled_skills=[],  # 按 workspace 的 enabled_skills 反推
    mcp_servers=mcp_servers,
    custom_agents=custom_agents,
    commands=commands,
    available_tools=None,  # 主 agent 全工具；sub-agent 各自白名单
    hooks={
        # 解析 [workspace_id=...] tag、抹 tag、注入 workspace context
        # 见 notebook 06
    },
    infinite_sessions={'enabled': True},
    system_message={'mode': 'append', 'content': 'You are the cowork agent.'},
)
print('SDK 一次 create_session 完成的事，等价于现在 cowork_worker 整个 hosted_agent_host.py + skills.py + tools.py')

## 6. 落到 cowork_worker 的迁移结论

| 现有抽象 | SDK 替代 | 删 / 改 |
|---|---|---|
| `SkillsContextProvider` | `skill_directories` + `disabled_skills` | **删** |
| `<available_files>` 系统块 | `infinite_sessions` + 内建 `view`/`find_files` | **删** |
| `make_tools()` 里 7-8 个文件/shell 工具 | 内建 `bash`/`view`/`edit_file`/`grep`/`find_files`/`fetch_url` | **删** |
| `python_tool`（rlimit 沙箱） | SDK 没对等沙箱 | **保留**（custom tool） |
| `save_artifact`/`load_artifact`（写 Azure Blob） | 业务定制 | **保留**（custom tool） |
| 单 agent + 多 prompt 分支 | 拆成 2-4 个 `custom_agents` | **重写**（更清晰） |
| `WorkspaceMiddleware` 解析 tag + chdir | `hooks.on_user_prompt_submitted` + `hooks.on_session_start` | **改** |
| `PER_TURN_MODEL` contextvar | `(workspace, model)` SessionPool | **改** |
| 自己存 history 到 Cosmos | `list_sessions` / `resume_session` + 仍存 Cosmos 做元数据索引 | **简化** |
| 前端 slash 指令 backend 拦截 | `commands=[...]` | **改**（可选） |

**净效果**：`worker/` 目录从 ~1500 行降到 **~300-400 行**（仅剩沙箱化 `python_tool`、artifact 存储、hooks、SessionPool、FastAPI 路由）。

至此 SDK 的 harness 能力全部演示完，下一步可以开干 `cowork_worker_sdk/` 骨架了。